In [0]:
from pyspark.sql.functions import current_timestamp, lit

CATALOG = "northwind_raw_data"
SCHEMA  = "source_kaggle_api"
VOLUME  = "vol_extracted_data"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

files = [
    "categories",
    "customers",
    "employees",
    "order_details",
    "orders",
    "products",
    "shippers",
]

# ============================================================
# FUNCTIONS
# ============================================================

def read_csv(file_name: str):
    """Reads a CSV file from the volume and returns a DataFrame."""
    path = f"{VOLUME_PATH}/{file_name}.csv"
    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", ",")
        .load(path)
    )


def add_metadata_columns(df, file_name: str):
    """Adds ingest metadata columns to the DataFrame"""
    return (
        df
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_path", lit(f"{VOLUME_PATH}/{file_name}.csv"))
    )


def write_bronze(df, file_name: str):
    """Writes the Dataframa as a Delta table in the raw schema (full refresh)."""
    target_table = f"{CATALOG}.{SCHEMA}.{file_name}"
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )
    print(f"✓ {target_table} ({df.count()} lines)")


# ============================================================
# LOOPING — Read, add metadata, and write to Bronze layer.
# ============================================================

print(f"Starting ingestion to {CATALOG}.{SCHEMA}...\n")

for file_name in files:
    try:
        df = read_csv(file_name)
        df = add_metadata_columns(df, file_name)
        write_bronze(df, file_name)
    except Exception as e:
        print(f"  ✗ Error processing '{file_name}': {e}")

print("\nIngestion completed.")